In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/odinaka60/PPE-detection.git
%cd PPE-detection

In [ ]:
!pip install ultralytics roboflow

In [ ]:
from google.colab import userdata

from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("aji-w7ttg").project("ppe-z0i9o")
version = project.version(3)
dataset = version.download("yolov8")
print("Dataset location:", dataset.location)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=80,
    imgsz=640,
    batch=16,
    device="0",
    patience=15,
    project="runs/train",
    name="ppe_v1",
    save_period=10,
)

print("Training complete.")
print("Best weights saved at: runs/train/ppe_v1/weights/best.pt")

In [ ]:
import shutil
from pathlib import Path

save_dir = Path("/content/drive/MyDrive/PPE-detection/models")
save_dir.mkdir(parents=True, exist_ok=True)

shutil.copy("runs/train/ppe_v1/weights/best.pt",
            save_dir / "best.pt")

print("Saved best.pt to Drive")

In [ ]:
best_model = YOLO("runs/train/ppe_v1/weights/best.pt")
best_model.export(format="onnx", imgsz=640, half=True)


shutil.copy(
    "runs/train/ppe_v1/weights/best.onnx",
    save_dir / "best.onnx"
)

print("Export complete. Files saved to Drive:")
print(str(save_dir / "best.pt"))
print(str(save_dir / "best.onnx"))